# Mask2Former on the smart-crops datasets (cabbage / purple kale / yellow onion)

Instance segmentation with **Mask2Former** (detectron2), trained separately on each of the
three crop datasets, so the numbers line up with the YOLO26s-seg runs.

**Kaggle settings required**
- Accelerator: **GPU T4 x2** or **GPU P100** (P100 is fine, only 1 GPU is used by default)
- **Internet: ON** (needed to pip-install detectron2 and download pretrained weights)
- Attach the `smart-crops` dataset

**What this notebook does**
1. Builds detectron2 + Mask2Former + the MSDeformAttn CUDA kernel (with the patches needed for torch 2.x).
2. Converts the YOLO-segmentation labels (`labels/*.txt` polygons) into COCO instance JSON.
3. Registers each crop dataset with detectron2 and fine-tunes `maskformer2_R50_bs16_50ep` from COCO weights.
4. Evaluates on the `test` split with COCO mask AP (`segm AP`, `AP50`, `AP75`) - directly comparable to `yolo segment val`.

**Timing:** ~15-20 min for the build, then ~1.5-3 h per dataset at the default 50 epochs.
Setup is a single session cost; if you only have time for one dataset per session, run the
setup cells then just the one train/eval pair you want (same pattern as the YOLO notebook).

In [ ]:
!nvidia-smi
import sys, torch, numpy, platform
print("python  ", platform.python_version())
print("torch   ", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())
print("numpy   ", numpy.__version__)
print("gpus    ", torch.cuda.device_count())

## 1. Build the environment

detectron2 and the MSDeformAttn kernel are compiled from source. This takes ~15 min and
must be re-run each session (Kaggle does not persist `site-packages`).

If you want to skip the rebuild in later sessions: after this cell succeeds once, run the
"save the build" cell at the bottom, publish `/kaggle/working/build_cache` as a Kaggle
dataset, and `pip install` the wheels from it instead.

In [ ]:
%%bash
set -e
cd /kaggle/working

pip install -q --no-cache-dir \
  ninja cython timm shapely h5py scikit-image \
  opencv-python-headless "pycocotools>=2.0.7" \
  fvcore iopath cloudpickle omegaconf hydra-core black tabulate

pip install -q --no-cache-dir "git+https://github.com/cocodataset/panopticapi.git"
echo "deps ok"

In [ ]:
%%bash
set -e
cd /kaggle/working

export TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6"
export FORCE_CUDA=1
export MAX_JOBS=2

rm -rf detectron2 d2repo          # kill the shadowing source dir first

if python -c "import detectron2; detectron2.__version__" 2>/dev/null; then
  echo "detectron2 already installed"
else
  git clone --depth 1 https://github.com/facebookresearch/detectron2.git d2repo
  pip install --no-build-isolation ./d2repo 2>&1 | tail -60
fi

cd /tmp && python -c "import detectron2; print('detectron2', detectron2.__version__)"

In [ ]:
%%bash
set -e
cd /kaggle/working

# Use your own fork here if you have local changes.
M2F_REPO=https://github.com/facebookresearch/Mask2Former.git

if [ ! -d Mask2Former ]; then
  git clone --depth 1 $M2F_REPO Mask2Former
fi
cd Mask2Former
pip install -q -r requirements.txt

# ---- patches for torch >= 1.11 / 2.x -------------------------------------
# THC/THCAtomics.cuh was deleted from pytorch; Tensor.type() is gone in torch 2.x.
cd mask2former/modeling/pixel_decoder/ops
sed -i 's|#include <THC/THCAtomics.cuh>|#include <ATen/cuda/Atomic.cuh>|' src/cuda/ms_deform_im2col_cuda.cuh
sed -i 's|\.type()\.is_cuda()|.is_cuda()|g' src/ms_deform_attn.h src/cuda/ms_deform_attn_cuda.cu src/cpu/ms_deform_attn_cpu.cpp
sed -i 's|value\.type()|value.scalar_type()|g' src/cuda/ms_deform_attn_cuda.cu
grep -n "Atomic.cuh" src/cuda/ms_deform_im2col_cuda.cuh || true

# ---- build the kernel ----------------------------------------------------
export TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6"
export FORCE_CUDA=1
export MAX_JOBS=4
pip install -q --no-build-isolation -e . || true
python -c "import MultiScaleDeformableAttention" 2>/dev/null || python setup.py build_ext --inplace
ls *.so 2>/dev/null || true
echo "MSDeformAttn built"

In [ ]:
import sys, importlib
sys.path.insert(0, "/kaggle/working/Mask2Former")
# last-resort path for a `build_ext --inplace` kernel
sys.path.append("/kaggle/working/Mask2Former/mask2former/modeling/pixel_decoder/ops")

import torch, detectron2
import MultiScaleDeformableAttention as MSDA
from mask2former import add_maskformer2_config
from detectron2.projects.deeplab import add_deeplab_config

print("detectron2", detectron2.__version__)
print("MSDeformAttn ok ->", MSDA.__file__)

# quick numerical smoke test of the kernel
from mask2former.modeling.pixel_decoder.ops.functions import MSDeformAttnFunction
print("kernel import ok")

## 2. Find the datasets

The three crop folders are YOLO-seg layout (`train/images`, `train/labels`, ...). The cell below
globs `/kaggle/input` for anything that looks like that and matches it to a crop by keyword, so it
works whether the mount is `/kaggle/input/smart-crops/...` or
`/kaggle/input/datasets/muhammadibnerafiq/smart-crops/...`.

If auto-detection picks the wrong folder, just hard-code `DATASETS[...]["root"]` in the next cell.

In [ ]:
import glob, os, json
from pathlib import Path

def discover_yolo_roots(search_root="/kaggle/input"):
    roots = set()
    for p in glob.glob(os.path.join(search_root, "**", "train", "images"), recursive=True):
        roots.add(str(Path(p).parent.parent))
    return sorted(roots)

ALL_ROOTS = discover_yolo_roots()
print("YOLO-style dataset roots found under /kaggle/input:")
for r in ALL_ROOTS:
    n = len(glob.glob(os.path.join(r, "train", "images", "*")))
    print(f"  {n:6d} train images   {r}")

In [ ]:
# keyword -> (dataset key, class name). Edit "root" below if auto-detect guessed wrong.
KEYWORDS = {
    "cabbage": ("cabbage", "cabbage"),
    "kale":    ("kale",    "purple_kale"),
    "onion":   ("onion",   "yellow_onion"),
}

def pick(keyword):
    hits = [r for r in ALL_ROOTS if keyword in r.lower()]
    if not hits:
        return None
    # deepest path wins (handles .../yellow_onion_extended/yellow_onion_extended)
    return max(hits, key=lambda r: (len(glob.glob(os.path.join(r, "train", "images", "*"))), len(r)))

DATASETS = {}
for kw, (key, cls) in KEYWORDS.items():
    root = pick(kw)
    DATASETS[key] = {"root": root, "names": [cls]}

for k, v in DATASETS.items():
    print(f"{k:8s} -> {v['root']}")

missing = [k for k, v in DATASETS.items() if v["root"] is None]
assert not missing, f"could not locate {missing}; set DATASETS[key]['root'] manually"

## 3. YOLO-seg labels -> COCO instance JSON

Each `labels/*.txt` line is `class x1 y1 x2 y2 ... xn yn` with normalised coordinates
(a 4-number line is treated as a `cx cy w h` box and turned into a rectangle polygon).
Images with no label file are kept with zero annotations; detectron2 drops them at train
time via `DATALOADER.FILTER_EMPTY_ANNOTATIONS`.

In [ ]:
import cv2
import numpy as np
from PIL import Image

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
COCO_ROOT = Path("/kaggle/working/coco")
COCO_ROOT.mkdir(parents=True, exist_ok=True)

def _poly_area(xs, ys):
    return 0.5 * abs(np.dot(xs, np.roll(ys, 1)) - np.dot(ys, np.roll(xs, 1)))

def parse_yolo_line(vals, W, H):
    """Return (flat_polygon, bbox_xywh, area) in absolute pixels, or None."""
    cls = int(float(vals[0]))
    c = [float(v) for v in vals[1:]]
    if len(c) == 4:                       # cx cy w h -> rectangle
        cx, cy, w, h = c
        x0, y0 = (cx - w / 2) * W, (cy - h / 2) * H
        x1, y1 = (cx + w / 2) * W, (cy + h / 2) * H
        xs = np.array([x0, x1, x1, x0]); ys = np.array([y0, y0, y1, y1])
    elif len(c) >= 6 and len(c) % 2 == 0:  # polygon
        pts = np.array(c, dtype=np.float64).reshape(-1, 2)
        xs, ys = pts[:, 0] * W, pts[:, 1] * H
    else:
        return None

    xs = np.clip(xs, 0, W - 1); ys = np.clip(ys, 0, H - 1)
    if len(xs) < 3:
        return None
    x0, y0, x1, y1 = xs.min(), ys.min(), xs.max(), ys.max()
    bw, bh = x1 - x0, y1 - y0
    area = _poly_area(xs, ys)
    if bw < 1 or bh < 1 or area < 1:
        return None
    poly = np.stack([xs, ys], axis=1).reshape(-1).tolist()
    return cls, [round(v, 2) for v in poly], [round(float(x0), 2), round(float(y0), 2),
                                              round(float(bw), 2), round(float(bh), 2)], float(area)

def yolo_split_to_coco(split_dir, class_names, out_json):
    split_dir = Path(split_dir)
    img_dir, lbl_dir = split_dir / "images", split_dir / "labels"
    images, annotations = [], []
    ann_id, n_skip = 1, 0

    files = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXT])
    for img_id, ip in enumerate(files, start=1):
        try:
            with Image.open(ip) as im:
                W, H = im.size
        except Exception:
            n_skip += 1
            continue
        images.append({"id": img_id, "file_name": ip.name, "width": W, "height": H})

        lp = lbl_dir / (ip.stem + ".txt")
        if not lp.exists():
            continue
        for line in lp.read_text().splitlines():
            vals = line.split()
            if len(vals) < 5:
                continue
            out = parse_yolo_line(vals, W, H)
            if out is None:
                n_skip += 1
                continue
            cls, poly, bbox, area = out
            annotations.append({
                "id": ann_id, "image_id": img_id, "category_id": cls + 1,
                "segmentation": [poly], "bbox": bbox, "area": area, "iscrowd": 0,
            })
            ann_id += 1

    coco = {
        "info": {"description": str(split_dir)},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": [{"id": i + 1, "name": n, "supercategory": "crop"}
                       for i, n in enumerate(class_names)],
    }
    Path(out_json).parent.mkdir(parents=True, exist_ok=True)
    with open(out_json, "w") as f:
        json.dump(coco, f)
    return len(images), len(annotations), n_skip

SPLITS = ["train", "valid", "test"]
REGISTRY = {}

for key, spec in DATASETS.items():
    REGISTRY[key] = {"names": spec["names"], "splits": {}}
    for split in SPLITS:
        sdir = Path(spec["root"]) / split
        if not (sdir / "images").is_dir():
            print(f"  [skip] {key}/{split} not present")
            continue
        out_json = COCO_ROOT / key / f"{split}.json"
        ni, na, ns = yolo_split_to_coco(sdir, spec["names"], out_json)
        REGISTRY[key]["splits"][split] = {"json": str(out_json), "images": str(sdir / "images"),
                                          "n_images": ni, "n_anns": na}
        print(f"  {key:8s} {split:6s} {ni:5d} images  {na:7d} instances  ({ns} skipped)")

with open(COCO_ROOT / "registry.json", "w") as f:
    json.dump(REGISTRY, f, indent=2)
print("\nwrote", COCO_ROOT / "registry.json")

In [ ]:
# sanity check: draw the converted polygons on 2 random training images
import random
import matplotlib.pyplot as plt
from pycocotools.coco import COCO

def preview(key, split="train", n=2, seed=0):
    info = REGISTRY[key]["splits"][split]
    coco = COCO(info["json"])
    ids = sorted(coco.getImgIds())
    random.Random(seed).shuffle(ids)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    axes = np.atleast_1d(axes)
    for ax, iid in zip(axes, ids[:n]):
        rec = coco.loadImgs([iid])[0]
        img = cv2.cvtColor(cv2.imread(os.path.join(info["images"], rec["file_name"])), cv2.COLOR_BGR2RGB)
        anns = coco.loadAnns(coco.getAnnIds(imgIds=[iid]))
        for a in anns:
            p = np.array(a["segmentation"][0], dtype=np.int32).reshape(-1, 2)
            cv2.polylines(img, [p], True, (255, 0, 0), 2)
        ax.imshow(img); ax.set_title(f"{key}/{split} {rec['file_name']} - {len(anns)} inst"); ax.axis("off")
    plt.tight_layout(); plt.show()

for k in REGISTRY:
    preview(k)

## 4. Register the datasets with detectron2

`train_net.py` has no hook for custom datasets, so we drop a `register_crops.py` next to it and
a thin `train_crops.py` launcher that imports it before calling the original `main()`.

In [ ]:
%%writefile /kaggle/working/Mask2Former/register_crops.py
import json
from detectron2.data import MetadataCatalog
from detectron2.data.datasets import register_coco_instances

REGISTRY_PATH = "/kaggle/working/coco/registry.json"


def register_all():
    with open(REGISTRY_PATH) as f:
        registry = json.load(f)
    for key, spec in registry.items():
        for split, info in spec["splits"].items():
            name = f"crops_{key}_{split}"
            if name in MetadataCatalog.list():
                continue
            register_coco_instances(name, {}, info["json"], info["images"])
            MetadataCatalog.get(name).thing_classes = spec["names"]
    return registry


REGISTRY = register_all()

In [ ]:
%%writefile /kaggle/working/Mask2Former/train_crops.py
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
sys.path.append(os.path.join(HERE, "mask2former", "modeling", "pixel_decoder", "ops"))

import register_crops  # noqa: F401  (registering happens on import)
from detectron2.engine import default_argument_parser, launch
from train_net import main

if __name__ == "__main__":
    args = default_argument_parser().parse_args()
    print("Command Line Args:", args)
    launch(
        main,
        args.num_gpus,
        num_machines=args.num_machines,
        machine_rank=args.machine_rank,
        dist_url=args.dist_url,
        args=(args,),
    )

In [ ]:
# COCO-pretrained Mask2Former R50 instance-segmentation weights (MODEL_ZOO.md)
!mkdir -p /kaggle/working/weights
!wget -q -nc -P /kaggle/working/weights \
  https://dl.fbaipublicfiles.com/maskformer/mask2former/coco/instance/maskformer2_R50_bs16_50ep/model_final_3c8ec9.pkl
!ls -lh /kaggle/working/weights

PRETRAINED = "/kaggle/working/weights/model_final_3c8ec9.pkl"

# Swin-T variant if you want a stronger backbone (needs ~2x the memory):
# https://dl.fbaipublicfiles.com/maskformer/mask2former/coco/instance/maskformer2_swin_tiny_bs16_50ep/model_final_86143f.pkl

## 5. Per-dataset config

Fine-tuning from COCO weights converges far faster than the YOLO runs - 50 epochs is plenty,
where YOLO needed 500 from a much smaller pretrain. Batch size 2 and `IMAGE_SIZE 640` fit a
single T4/P100; raise `IMS_PER_BATCH` on a bigger GPU and scale `BASE_LR` with it.

In [ ]:
M2F_DIR = "/kaggle/working/Mask2Former"
RUNS = "/kaggle/working/runs_m2f"
CFG_DIR = "/kaggle/working/cfgs"
os.makedirs(CFG_DIR, exist_ok=True)

BASE_CFG = f"{M2F_DIR}/configs/coco/instance-segmentation/maskformer2_R50_bs16_50ep.yaml"

EPOCHS = 50
IMS_PER_BATCH = 2
BASE_LR = 5e-5          # 1e-4 is the bs16 LR; we run bs2 and fine-tune, so lower
IMAGE_SIZE = 640
NUM_GPUS = 1            # keep at 1: register_crops runs in-process only for single-GPU launch

def make_cfg(key, epochs=EPOCHS, ims_per_batch=IMS_PER_BATCH, base_lr=BASE_LR,
             image_size=IMAGE_SIZE, weights=PRETRAINED):
    spec = REGISTRY[key]
    n_train = spec["splits"]["train"]["n_images"]
    n_cls = len(spec["names"])
    max_iter = max(1000, int(round(epochs * n_train / ims_per_batch)))
    steps = [int(max_iter * 0.75), int(max_iter * 0.9)]
    period = max(500, max_iter // 5)
    out_dir = f"{RUNS}/{key}"

    val_split = "valid" if "valid" in spec["splits"] else "test"
    yaml_text = f"""_BASE_: {BASE_CFG}
DATASETS:
  TRAIN: ("crops_{key}_train",)
  TEST: ("crops_{key}_{val_split}",)
MODEL:
  WEIGHTS: "{weights}"
  SEM_SEG_HEAD:
    NUM_CLASSES: {n_cls}
SOLVER:
  IMS_PER_BATCH: {ims_per_batch}
  BASE_LR: {base_lr}
  MAX_ITER: {max_iter}
  STEPS: [{steps[0]}, {steps[1]}]
  CHECKPOINT_PERIOD: {period}
  WARMUP_ITERS: 100
  AMP:
    ENABLED: True
INPUT:
  IMAGE_SIZE: {image_size}
  MIN_SCALE: 0.5
  MAX_SCALE: 1.5
  MIN_SIZE_TEST: {image_size}
  MAX_SIZE_TEST: {int(image_size * 1.6)}
TEST:
  EVAL_PERIOD: {period}
  DETECTIONS_PER_IMAGE: 100
DATALOADER:
  NUM_WORKERS: 2
  FILTER_EMPTY_ANNOTATIONS: True
SEED: 42
OUTPUT_DIR: "{out_dir}"
"""
    path = f"{CFG_DIR}/{key}.yaml"
    with open(path, "w") as f:
        f.write(yaml_text)
    print(f"{key}: {n_train} train images -> MAX_ITER={max_iter} (~{epochs} epochs), out={out_dir}")
    return path

CFGS = {k: make_cfg(k) for k in REGISTRY}

In [ ]:
def train(key, resume=False, extra=""):
    cmd = (f"cd {M2F_DIR} && python train_crops.py --config-file {CFGS[key]} "
           f"--num-gpus {NUM_GPUS}" + (" --resume" if resume else "") +
           (f" {extra}" if extra else ""))
    print(cmd)
    return cmd

def evaluate(key, split="test", weights=None, extra=""):
    weights = weights or f"{RUNS}/{key}/model_final.pth"
    out = f"{RUNS}/{key}/eval_{split}"
    cmd = (f"cd {M2F_DIR} && python train_crops.py --config-file {CFGS[key]} "
           f"--num-gpus 1 --eval-only "
           f"MODEL.WEIGHTS {weights} "
           f"DATASETS.TEST '(\"crops_{key}_{split}\",)' "
           f"OUTPUT_DIR {out}" + (f" {extra}" if extra else ""))
    print(cmd)
    return cmd

print("helpers ready")

## 6. Train + evaluate

One pair of cells per dataset. Comment out the ones you are not running this session -
Kaggle caps a session at 12 h and each dataset takes roughly 1.5-3 h.

Add `--resume` to a training cell to pick up from the last checkpoint after a session restart
(checkpoints live in `/kaggle/working/runs_m2f/<crop>`, which does persist between commits).

In [ ]:
# ---------------- cabbage ----------------
#cmd = train("cabbage")
#!{cmd}

In [ ]:
#cmd = evaluate("cabbage", "test")
#!{cmd}

In [ ]:
# ---------------- purple kale ----------------
# cmd = train("kale")
# !{cmd}

In [ ]:
# cmd = evaluate("kale", "test")
# !{cmd}

In [ ]:
# ---------------- yellow onion ----------------
cmd = train("onion")
!{cmd}

In [ ]:
cmd = evaluate("onion", "test")
!{cmd}

## 7. Results

Recomputes COCO mask AP from the saved predictions so the table does not depend on scraping
the training log. `AP` is mAP@50-95 and `AP50` is mAP@50 - the same quantities
`yolo segment val` prints as `mAP50-95(M)` and `mAP50(M)`.

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import contextlib, io as _io

def coco_eval(key, split="test", iou_type="segm"):
    gt_json = REGISTRY[key]["splits"][split]["json"]
    hits = glob.glob(f"{RUNS}/{key}/eval_{split}/**/coco_instances_results.json", recursive=True)
    if not hits:
        return None
    gt = COCO(gt_json)
    with contextlib.redirect_stdout(_io.StringIO()):
        dt = gt.loadRes(hits[0])
        e = COCOeval(gt, dt, iou_type)
        e.evaluate(); e.accumulate()
    e.summarize()
    s = e.stats
    return {"AP": s[0], "AP50": s[1], "AP75": s[2],
            "APs": s[3], "APm": s[4], "APl": s[5], "AR100": s[8]}

rows = []
for key in REGISTRY:
    for it in ("segm", "bbox"):
        r = coco_eval(key, "test", it)
        if r:
            rows.append({"dataset": key, "type": it, **{k: round(v * 100, 2) for k, v in r.items()}})

import pandas as pd
df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/mask2former_results.csv", index=False)
df

In [ ]:
# qualitative predictions on a few test images
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.data import MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.projects.deeplab import add_deeplab_config
from mask2former import add_maskformer2_config

for key, spec in REGISTRY.items():
    for split, info in spec["splits"].items():
        name = f"crops_{key}_{split}"
        if name not in MetadataCatalog.list():
            register_coco_instances(name, {}, info["json"], info["images"])
            MetadataCatalog.get(name).thing_classes = spec["names"]

def show_predictions(key, split="test", n=3, score_thresh=0.5, seed=0):
    cfg = get_cfg()
    add_deeplab_config(cfg); add_maskformer2_config(cfg)
    cfg.merge_from_file(CFGS[key])
    cfg.MODEL.WEIGHTS = f"{RUNS}/{key}/model_final.pth"
    cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    cfg.freeze()
    predictor = DefaultPredictor(cfg)
    meta = MetadataCatalog.get(f"crops_{key}_{split}")

    info = REGISTRY[key]["splits"][split]
    files = sorted(os.listdir(info["images"]))
    random.Random(seed).shuffle(files)

    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    for ax, fn in zip(np.atleast_1d(axes), files[:n]):
        im = cv2.imread(os.path.join(info["images"], fn))
        out = predictor(im)
        inst = out["instances"].to("cpu")
        inst = inst[inst.scores > score_thresh]
        v = Visualizer(im[:, :, ::-1], metadata=meta, scale=1.0, instance_mode=ColorMode.IMAGE)
        ax.imshow(v.draw_instance_predictions(inst).get_image())
        ax.set_title(f"{key} {fn} - {len(inst)} inst"); ax.axis("off")
    plt.tight_layout(); plt.show()
    del predictor; torch.cuda.empty_cache()

show_predictions("cabbage")
# show_predictions("kale")
# show_predictions("onion")

## 8. Trim the output directory

Same idea as `keep_only_best()` in the YOLO notebook - Kaggle refuses to commit an output
larger than ~20 GB, and Mask2Former checkpoints are ~200 MB each.

In [ ]:
def keep_only_final(root=RUNS):
    removed, freed = 0, 0
    for f in glob.glob(f"{root}/**/*.pth", recursive=True):
        if os.path.basename(f) == "model_final.pth":
            continue
        freed += os.path.getsize(f); os.remove(f); removed += 1
        print("removed:", f)
    for f in glob.glob(f"{root}/**/instances_predictions.pth", recursive=True):
        if os.path.exists(f):
            freed += os.path.getsize(f); os.remove(f); removed += 1
            print("removed:", f)
    print(f"\n{removed} files removed, {freed/1e6:.1f} MB freed")
    for k in sorted(glob.glob(f"{root}/**/model_final.pth", recursive=True)):
        print(" kept", k, f"({os.path.getsize(k)/1e6:.1f} MB)")

keep_only_final()
!du -sh /kaggle/working